# Milestone 5 lab, notebook 8 of 8: The Monday answer

## Where we are

| Notebook | What you do |
|---|---|
| 1. Did it run? | Read Airflow's run history and the pipeline's own. Run one night twice |
| 2. On time, as a number | Measure freshness. Turn "on time" into a number with a target |
| 3. Five signals | Monitoring asks if the job ran. Observability asks if the data is right |
| 4. Tell the person who can act | Airflow calls back, and the right person gets an alert |
| 5. Too many alerts | Count the careless way. Then group, send once, rank and silence |
| 6. Fix one day | Fix the source, run one night again, prove the number |
| 7. Four drills | Four faults on a clean day, each caught and repaired. Write an incident note |
| **8. The Monday answer** (you are here) | The whole week, the recovery, and the answer to the question |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. Airflow's page is http://localhost:8080 .

**About ten minutes of reading, and six minutes of waiting** while Airflow runs the week and the recovery, and the tests run.

In [1]:
import subprocess
import sys

import pandas as pd
from lab_helpers import (
    LAST_NIGHT, MILESTONE_DIR, WEEK, airflow_tools, answer_key_week, backfill, bonus_report,
    health_numbers, load_result, money, night_by_night, open_alerts, run_nights, save_result,
    show_health, show_report, sql, start_again,
)

## Step 1: The question

Monday 31 August, 09:00. The head of retail operations asks the question of this whole project:

> Can I pay the bonus from last week's net sales, store by store? And if I cannot trust a number, which one, and why?

Six managers wait: Marta Lindqvist, Dana Okafor, Tomas Reyes, Ines Marchetti, Samir Haddad and Aiko Tanaka. In milestone 4 the answer was right, and late: 1 of 6.

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, what happens when it fails, and who is told.

Run the bonus week one last time, from nothing, the way it really ran: Airflow, seven nights.

In [2]:
start_again()
run_nights(WEEK)

Empty: the warehouse of this milestone, the stores' landing folder, the alerts log,
and Airflow's runs of lotus_nightly_m5.


,night,tasks,succeeded,seconds,stores that can be paid
0,Mon 08-24,8,8,19,5
1,Tue 08-25,8,8,24,4
2,Wed 08-26,8,8,19,4
3,Thu 08-27,8,8,24,2
4,Fri 08-28,8,8,23,3
5,Sat 08-29,8,8,19,2
6,Sun 08-30,8,8,24,1


## Step 2: Monday morning, before anyone fixes anything

In [3]:
show_report(LAST_NIGHT)

Lotus Retails. Store managers' bonus, week of Mon 2026-08-24
Net sales from Mon 2026-08-24 to Sun 2026-08-30

  Store                           Manager              Net sales  Bonus (1%)   Pay?
  S01  Lotus Ashby High Street    Marta Lindqvist      $5,153.25      $51.53   yes
  S02  Lotus Ashby Market Square  Dana Okafor          $5,724.68           -   DO NOT PAY YET
         Sun 08-30: 4 rows set aside ($23.95): Nothing is sold while the store is closed
  S03  Lotus Calder Bridge        Tomas Reyes          $5,097.11           -   DO NOT PAY YET
         Mon 08-24: 1 row set aside ($9.49): A line adds up; A refund gives money back
  S04  Lotus Calder Station       Ines Marchetti       $4,585.57           -   DO NOT PAY YET
         Tue 08-25: no sales file arrived. The tills counted $587.42
  S05  Lotus Fenwick Green        Samir Haddad         $3,591.53           -   DO NOT PAY YET
         Thu 08-27: 1 row set aside ($1,522.20): A weight is a believable weight
  S06  Lotus Fenwick 

The same report as milestone 4: 1 of 6. What is different is everything that happened during the week. Each problem was sent, once, to the person who could act, on its night:

In [4]:
week_alerts = night_by_night()
sent_per_night = [int(n) for n in week_alerts["sent: new tickets"][:7]]
week_alerts[["night", "shout: one per failed test", "sent: new tickets", "resolved"]]

,night,shout: one per failed test,sent: new tickets,resolved
0,Mon 08-24,3,1,0
1,Tue 08-25,7,2,0
2,Wed 08-26,8,0,0
3,Thu 08-27,9,2,0
4,Fri 08-28,14,3,1
5,Sat 08-29,14,1,1
6,Sun 08-30,15,1,0
7,the week,70,10,2


## Step 3: Four stores send corrected files

The tickets went out on the night. So the stores had the week to answer. Four did:

- `S03`: the refund with a lost minus sign, Monday.
- `S05`: the bananas weighed in grams, Thursday.
- `S06`: three sales of a product that was not in the product list, Saturday.
- `S02`: four sales stamped 03:10 by a till with a wrong clock, Sunday.

`S04`'s Tuesday file never comes. That is realistic: an export that failed six days ago may be gone.

In [5]:
for case in (1, 3, 5, 6):
    print(airflow_tools.stores("fix", "--case", str(case)).splitlines()[0])

Fixed at the source: case 1, A refund stored as a positive amount.
Fixed at the source: case 3, A weight entered in grams, not kilograms.


Fixed at the source: case 5, A sale at 03:10 in a store that closes at 22:00.
Fixed at the source: case 6, A sale for a product that is not in products.


## Step 4: Backfill only the four nights

Monday, Thursday, Saturday and Sunday. Saturday and Sunday are next to each other, so they are one backfill. Three commands, four nights, oldest first. The last night run is Sunday, so the outputs end in Sunday's state.

In [6]:
pd.DataFrame(
    backfill("2026-08-24", "2026-08-24")
    + backfill("2026-08-27", "2026-08-27")
    + backfill("2026-08-29", "2026-08-30")
)

,day,state
0,2026-08-24,success
1,2026-08-27,success
2,2026-08-29,success
3,2026-08-30,success


## Step 5: The answer

In [7]:
show_report(LAST_NIGHT)

Lotus Retails. Store managers' bonus, week of Mon 2026-08-24
Net sales from Mon 2026-08-24 to Sun 2026-08-30

  Store                           Manager              Net sales  Bonus (1%)   Pay?
  S01  Lotus Ashby High Street    Marta Lindqvist      $5,153.25      $51.53   yes
  S02  Lotus Ashby Market Square  Dana Okafor          $5,748.63      $57.49   yes
  S03  Lotus Calder Bridge        Tomas Reyes          $5,087.62      $50.88   yes
  S04  Lotus Calder Station       Ines Marchetti       $4,585.57           -   DO NOT PAY YET
         Tue 08-25: no sales file arrived. The tills counted $587.42
  S05  Lotus Fenwick Green        Samir Haddad         $3,593.05      $35.93   yes
  S06  Lotus Fenwick Harbour      Aiko Tanaka          $4,418.89      $44.19   yes

  5 of 6 stores can be paid.
  Tonight's gates: 38 of 49 tests passed, 7 warnings.


Five stores can be paid. Hold each number against the answer key, which was added up from the receipts, not by the pipeline:

In [8]:
final = bonus_report(LAST_NIGHT)
pd.DataFrame([
    {
        "store": s["store_id"],
        "manager": s["manager_name"],
        "pay": "yes" if s["pay"] else "no",
        "net sales": money(s["net_sales_cents"]),
        "answer key": money(answer_key_week(s["store_id"])["net_sales_cents"]),
        "equal": s["net_sales_cents"] == answer_key_week(s["store_id"])["net_sales_cents"],
        "bonus": money(s["bonus_cents"]) if s["pay"] else "-",
    }
    for s in final["stores"]
])

,store,manager,pay,net sales,answer key,equal,bonus
0,S01,Marta Lindqvist,yes,"$5,153.25","$5,153.25",True,$51.53
1,S02,Dana Okafor,yes,"$5,748.63","$5,748.63",True,$57.49
2,S03,Tomas Reyes,yes,"$5,087.62","$5,087.62",True,$50.88
3,S04,Ines Marchetti,no,"$4,585.57","$5,172.99",False,-
4,S05,Samir Haddad,yes,"$3,593.05","$3,593.05",True,$35.93
5,S06,Aiko Tanaka,yes,"$4,418.89","$4,418.89",True,$44.19


Five stores: yes, and each number equals the answer key to the cent. One store: no. `S04` is $587.42 short, because one file never arrived, and the report says so.

## Step 6: Health, and what is still open

In [9]:
show_health(LAST_NIGHT)

Is the pipeline healthy? Week of Mon 2026-08-24, up to Sun 2026-08-30

  ok   Did it run?              7 of 7 nights published
  ok   Did it finish on time?   the slowest night took 5 seconds. Target: under 300
  MISS Is the data fresh?       39 of 41 sales files arrived by 01:00: 95.1%. Target: 98%
         late:    S06 2026-08-27, 23.7 hours after it was due
         missing: S04 2026-08-25
       Did every store report?  40 of 41 store days can be paid from: 97.6%


The late file is still late, and the missing file is still missing: a recovery does not rewrite history. But now 40 of 41 store days can be paid from.

In [10]:
open_alerts("ticket")

,level,to_whom,title,first_night
0,ticket,Head of retail operations,S04 Tue 08-25: No sales file arrived,2026-08-25
1,ticket,"Finance, and the buying team",The file supplier_prices_SUP-B_2026-08-25.csv was refused,2026-08-25
2,ticket,Finance,S04 Fri 08-28 charged a price that is not in the price list,2026-08-28
3,ticket,"Finance, and the buying team",The file supplier_prices_SUP-C_2026-08-28.csv was refused,2026-08-28
4,ticket,The replenishment team,S01 Sun 08-30 has a product with no stock count,2026-08-30
5,ticket,The replenishment team,S02 Sun 08-30 has a product with no stock count,2026-08-30
6,ticket,The replenishment team,S03 Sun 08-30 has a product with no stock count,2026-08-30
7,ticket,The replenishment team,S04 Sun 08-30 has a product with no stock count,2026-08-30
8,ticket,The replenishment team,S05 Sun 08-30 has a product with no stock count,2026-08-30
9,ticket,The replenishment team,S06 Sun 08-30 has a product with no stock count,2026-08-30


One ticket about a store's pay is still open, and it says the same thing as the report: `S04` Tue 08-25, no sales file arrived.

The others do not stop a bonus. They are somebody else's work for Monday: two supplier files that were refused, and a store that charged yesterday's price, for finance and the buying team. And six tickets to the replenishment team that were not there before the recovery. Why?

## Step 7: A fix that made a small new problem

In [11]:
sql("""
    SELECT o.store_id, o.sales_date AS count_missing_for, o.product_id, p.product_name, o.note
    FROM audit.order_without_a_count o
    JOIN clean.products p ON p.product_id = o.product_id
    ORDER BY o.store_id
""")

,store_id,count_missing_for,product_id,product_name,note
0,S01,2026-08-30,P-041,Pumpkin soup 500ml,no count: order by hand
1,S02,2026-08-30,P-041,Pumpkin soup 500ml,no count: order by hand
2,S03,2026-08-30,P-041,Pumpkin soup 500ml,no count: order by hand
3,S04,2026-08-30,P-041,Pumpkin soup 500ml,no count: order by hand
4,S05,2026-08-30,P-041,Pumpkin soup 500ml,no count: order by hand
5,S06,2026-08-30,P-041,Pumpkin soup 500ml,no count: order by hand


`P-041` is Pumpkin soup: the product of `S06`'s Saturday, the one that was missing from the product list. Fixing it at the source meant the buying team typed it into the list. So the order list for Monday now includes it, at every store. And no shelf has ever been counted for it.

The pipeline did not guess. It wrote "no count: order by hand", and told the team who orders. A fix can make a small new problem. A good pipeline says so on the night, to the right person.

## Step 8: Save it, and test it

Save the final answer. `tests/test_notebook_parity.py` runs the same recovery without Airflow and holds the production code against what you see here.

In [12]:
health = health_numbers(LAST_NIGHT)
save_result(
    "answer_key_monday",
    {
        "night": LAST_NIGHT.isoformat(),
        "stores": {
            s["store_id"]: {
                "pay": s["pay"],
                "net_sales_cents": s["net_sales_cents"],
                "bonus_cents": s["bonus_cents"],
                "why_not": s["why_not"],
            }
            for s in final["stores"]
        },
        "can_be_paid": [s["store_id"] for s in final["stores"] if s["pay"]],
        "health": {
            key: health[key]
            for key in ("nights", "nights_published", "files_expected", "files_on_time",
                        "files_on_time_percent", "files_on_time_target", "late_files",
                        "missing_files", "store_days", "store_days_ok")
        },
        "alerts_sent_per_night": sent_per_night,
    },
)

Saved notebooks/results/answer_key_monday.json


In [13]:
saved = load_result("answer_key_monday")
print("Can be paid:          ", ", ".join(saved["can_be_paid"]))
print("Do not pay yet:       ", {s: v["why_not"] for s, v in saved["stores"].items() if not v["pay"]})
print("Alerts sent per night:", saved["alerts_sent_per_night"], "=", sum(saved["alerts_sent_per_night"]))

Can be paid:           S01, S02, S03, S05, S06
Do not pay yet:        {'S04': ['Tue 08-25: no sales file arrived. The tills counted $587.42']}
Alerts sent per night: [1, 2, 0, 2, 3, 1, 1] = 10


`tests/test_know_alert_recover.py` checks everything this milestone showed you: the run history, the four health numbers, the ten tickets against seventy, the routing, the four drills, and this answer. It takes a minute or two.

In [14]:
done = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_know_alert_recover.py", "-q", "--no-header",
     "--color=no", "-p", "no:cacheprovider"],
    cwd=MILESTONE_DIR, capture_output=True, text=True,
)
print(done.stdout.strip().splitlines()[-1])

34 passed in 82.38s (0:01:22)


## Step 9: Nothing is still wrong

Every milestone so far ended with something still wrong. This one does not. The question has its answer:

> **Yes for five stores. No for one. `S04`: Tue 08-25, no sales file arrived, the tills counted $587.42.**

There is more a data team can do. It belongs to other projects of this series, or to tools that need an account:

| Not built here | Where it belongs |
|---|---|
| Building a pipeline from the start, and the full story of Airflow | Daily Orders |
| Checks on events as they arrive, one by one | The clickstream project |
| A data lake, and table formats | The lakehouse project |
| A dashboard for the health numbers | The warehouse project |
| AI | Retail Profit Stack |

And what a real team would add to this pipeline, with the same decisions you made here:

- A **paging service**, such as PagerDuty, and Slack or email, at the one line where an alert is written.
- A **quality tool**, such as Great Expectations or Soda, if the team already uses one. The rules stay the same.
- A **schema registry**, where every sender's columns are agreed and versioned, instead of a contract file written by hand.
- **Lineage tools**, which draw the map of what reads what from the SQL, instead of a map written by hand.
- **Machine-learned anomaly detection**, when a normal-day average is not enough: holidays that move, slow drift, hundreds of stores.

## What you now know

- The pipeline knows whether it ran, on time, with every store, as numbers with targets.
- It tells the person who can act, once, on the night, with what to do.
- It recovers: fix the source, backfill only the nights that were wrong, and prove the number.
- The answer to the Monday question: yes for five stores, no for one, and here is the row that says why.

**Say it in an interview:** "At the end of the week one store of six could be paid, and every manager who waited had been told on the night. Four stores sent corrected files. I backfilled only their four nights, and five stores could be paid, each to the cent of the answer key. The sixth could not, and the report said exactly why: one file that never came."

**The theory:** [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), [Incremental Processing & Backfills](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/incremental-processing-and-backfills), [Airflow: Monitoring, Logging, Alerting & Production Failures](https://www.d8loop.com/modules/knowledge-corner/airflow/monitoring-logging-alerting-and-production-failures), and [Airflow: Backfills, Catchup, Retries & SLA](https://www.d8loop.com/modules/knowledge-corner/airflow/backfills-catchup-retries-and-sla).

## Before you go

- Open the [milestone 5 flow page](https://vijaysngrajput.github.io/quality-gates-on-a-running-pipeline/src/m5_know_alert_recover/milestone_5_flow.html). Watch a night go through the gates, the alerts go out, and a corrected file come back.
- Open the [milestone 5 recap page](https://vijaysngrajput.github.io/quality-gates-on-a-running-pipeline/src/m5_know_alert_recover/milestone_5_recap.html) for a look back over the milestone, and three questions.
- The [check cards page](https://vijaysngrajput.github.io/quality-gates-on-a-running-pipeline/checks/check_cards.html) shows every table's checks side by side.
- Stop everything you started, from the project folder: `docker compose down`. Your data is kept. To delete it too: `docker compose down -v`.